In [11]:
import requests
import json
from bs4 import BeautifulSoup
from pathlib import Path

In [6]:
data_urls = [
    ("https://spring.io/guides/gs/relational-data-access",       "Accessing Relational Data using JDBC with Spring"),
    ("https://spring.io/guides/gs/accessing-data-neo4j",         "Accessing Data with Neo4j"),
    ("https://spring.io/guides/gs/managing-transactions",        "Managing Transactions"),
    ("https://spring.io/guides/gs/accessing-data-jpa",           "Accessing Data with JPA"),
    ("https://spring.io/guides/gs/accessing-data-mongodb",       "Accessing Data with MongoDB"),
    ("https://spring.io/guides/gs/accessing-data-rest",          "Accessing JPA Data with REST"),
    ("https://spring.io/guides/gs/accessing-mongodb-data-rest",  "Accessing MongoDB Data with REST"),
    ("https://spring.io/guides/gs/accessing-data-mysql",         "Accessing data with MySQL"),
    ("https://spring.io/guides/gs/spring-data-reactive-redis",   "Accessing Data Reactively with Redis"),
    ("https://spring.io/guides/gs/accessing-data-r2dbc",         "Accessing data with R2DBC"),
    ("https://spring.io/guides/gs/accessing-data-cassandra",     "Accessing Data with Cassandra"),
]
urls = data_urls

In [9]:
#security documentations
security_urls = [
    ("https://spring.io/guides/gs/authenticating-ldap", "Authenticating a User with LDAP"),
    ("https://spring.io/guides/gs/securing-web", "Securing a Web Application"),
    ("https://spring.io/guides/gs/rest-service-cors", "Enabling Cross Origin Requests for a RESTful Web Service"),
    ("https://spring.io/guides/tutorials/spring-security-and-angular-js", "Spring Security and Angular"),
    ("https://spring.io/guides/tutorials/spring-boot-oauth2", "Spring Boot and OAuth2"),
    ("https://spring.io/guides/gs/accessing-vault", "Accessing Vault"),
    ("https://spring.io/guides/gs/vault-config", "Vault Configuration")
]
urls = security_urls

In [12]:


for url, title in urls:
    response = requests.get(url)
    if response.status_code != 200:
        print(f"FAILED ({response.status_code}): {url}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")
    for tag in soup.find_all(['nav', 'header', 'footer', 'script', 'style']):
        tag.decompose()

    sections = []
    current = {'heading': '', 'content': ''}

    for tag in soup.find_all(['h2', 'h3', 'p', 'pre', 'li']):
        if tag.name in ['h2', 'h3']:
            if current['content'].strip():
                sections.append({
                    'heading': current['heading'],
                    'content': current['content'].strip(),
                    'title':   title,
                    'source':  url,
                })
            current = {'heading': tag.get_text(strip=True), 'content': ''}
        else:
            current['content'] += tag.get_text(separator=' ', strip=True) + '\n'

    if current['content'].strip():
        sections.append({
            'heading': current['heading'],
            'content': current['content'].strip(),
            'title':   title,
            'source':  url,
        })

    guide_data[title] = sections
    print(f"OK  {title}  ({len(sections)} sections)")

print(f"\nTotal guides scraped: {len(guide_data)}")

OK  Authenticating a User with LDAP  (13 sections)
OK  Securing a Web Application  (11 sections)
OK  Enabling Cross Origin Requests for a RESTful Web Service  (16 sections)
OK  Spring Security and Angular  (64 sections)
OK  Spring Boot and OAuth2  (27 sections)
OK  Accessing Vault  (16 sections)
OK  Vault Configuration  (14 sections)

Total guides scraped: 7


In [13]:
output_dir = Path("../data/json/security")
output_dir.mkdir(parents=True, exist_ok=True)

for title, sections in guide_data.items():
    filename = title.replace(' ', '_') + '.json'
    path = output_dir / filename
    path.write_text(json.dumps(sections, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved: {path}  ({len(sections)} sections)")

Saved: ..\data\json\security\Authenticating_a_User_with_LDAP.json  (13 sections)
Saved: ..\data\json\security\Securing_a_Web_Application.json  (11 sections)
Saved: ..\data\json\security\Enabling_Cross_Origin_Requests_for_a_RESTful_Web_Service.json  (16 sections)
Saved: ..\data\json\security\Spring_Security_and_Angular.json  (64 sections)
Saved: ..\data\json\security\Spring_Boot_and_OAuth2.json  (27 sections)
Saved: ..\data\json\security\Accessing_Vault.json  (16 sections)
Saved: ..\data\json\security\Vault_Configuration.json  (14 sections)
